In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 1 · Environment setup (Shift+Enter to run)                       │
# │                                                                        │
# │  Welcome! This notebook runs GEMC directly in your browser.            │
# │  How to use:                                                           │
# │                                                                        │
# │  1. Click any grey code cell to select it                              │
# │  2. Press `Shift + Enter` to run it (or click Run in the toolbar)      │
# │  3. Run cells in order, top to bottom                                  │
# │  4. Wait for `In [*]` to change to a number before continuing          │
# │                                                                        │
# │  Optional cells are provided to edit the code and the YAML files.      │
# │  After editing, re-run the other cells to see the changes.             │
# │                                                                        │
# │  Documentation:                                                        │
# │  https://gemc.github.io/home/documentation/quickstart/                 │
# │                                                                        │
# │  This notebook creates a small counter detector from the GEMC system   │
# │  template, builds its geometry, runs GEMC, and plots the output.       │
# │                                                                        │
# │  Import notebook helpers                                               │
# │  Create the quickstart counter files                                   │
# └────────────────────────────────────────────────────────────────────────┘

import os
import subprocess
import sys
from pathlib import Path


if Path.cwd().name == "counter":
    os.chdir(Path.cwd().parent)

notebooks_dir = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebook_tools").is_dir()
)
sys.path.insert(0, str(notebooks_dir))
from notebook_tools import edit, run_gemc_display

from pygemc.api.run_geometry import run_geometry

subprocess.run(["rm", "-rf", "counter"], check=True)
subprocess.run(["gemc-system-template", "-s", "counter"], check=True)
os.chdir("counter")

result = subprocess.run(["ls", "-l"], capture_output=True, text=True, check=True)
print(result.stdout)


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 2 · Build the counter geometry (Shift+Enter to run)              │
# │                                                                        │
# │  Print counter.py, write gemc.db, and render the geometry with         │
# │  PyVista. Use the mouse to zoom, rotate, or pan the view.              │
# └────────────────────────────────────────────────────────────────────────┘

print(Path("counter.py").read_text())
run_geometry("counter.py")


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 3 · Run 100 events in GEMC (Shift+Enter to run)                  │
# │                                                                        │
# │  Write CSV and JSON output, then display the off-screen GEMC image.    │
# │  If GEMC fails during startup, re-run this cell.                       │
# └────────────────────────────────────────────────────────────────────────┘

driver = '-g4view=[{driver: TOOLSSG_OFFSCREEN}]'
camera = '-g4camera=[{phi: -10*deg, theta: 250*deg}]'
light = '-g4light=[{phi: 160*deg, theta: 120*deg}]'

result = subprocess.run(
    [
        "gemc",
        "counter.yaml",
        driver,
        camera,
        light,
        "-n=100",
        "-nthreads=1",
    ],
    capture_output=True,
    text=True,
)
run_gemc_display(result)

In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 4 · Plot total energy deposited (Shift+Enter to run)             │
# │                                                                        │
# │  Display a log-scale 50-bin histogram from the CSV output.             │
# └────────────────────────────────────────────────────────────────────────┘

from pygemc import plot_variable, read_output

plot_variable(
    read_output("counter_t0_digitized.csv"),
    "totEdep",
    bins=50,
    logy=True,
    show=True,
)



In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 5 · Show JSON output (Shift+Enter to run)                        │
# │                                                                        │
# │  Show the first 25 lines of counter_t0.json.                           │
# └────────────────────────────────────────────────────────────────────────┘

json_file = Path("counter_t0.json")
print("".join(json_file.read_text().splitlines(True)[:25]))


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Optional: Cell 6 · Edit counter.py (Shift+Enter to run)               │
# │                                                                        │
# │  After saving changes, re-run cells 2-5.                               │
# └────────────────────────────────────────────────────────────────────────┘

edit("counter.py")


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Optional: Cell 7 · Edit geometry.py (Shift+Enter to run)              │
# │                                                                        │
# │  After saving changes, re-run cells 2-5.                               │
# └────────────────────────────────────────────────────────────────────────┘

edit("geometry.py")


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Optional: Cell 8 · Edit materials.py (Shift+Enter to run)             │
# │                                                                        │
# │  After saving changes, re-run cells 2-5.                               │
# └────────────────────────────────────────────────────────────────────────┘

edit("materials.py")


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Optional: Cell 9 · Edit counter.yaml (Shift+Enter to run)             │
# │                                                                        │
# │  Change the generator or output, then re-run cells 3-5.                │
# └────────────────────────────────────────────────────────────────────────┘

edit("counter.yaml")
